### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 0.1_cast_splitting_to_netcdf
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es) & Bàrbara Barceló-Llull
### Data from BioSWOT-Med experiment

# 
**DESCRIPTION**:
 This script reads raw MVP (Moving Vessel Profiler) data files (.m1 format), 
 parses their internal metadata (NMEA coordinates, date, time), and splits 
 the continuous data stream into individual 'downcast' and 'upcast' profiles.
 Finally, it exports the separated profiles as NetCDF (.nc) files, ready for 
 further quality control and processing. (modified from a code to read MVP 
 data developed by Eugenio Cutolo)
#
 INPUT: Directory containing raw .m1 files.
#
 OUTPUT: Directory containing parsed _downcast.nc and _upcast.nc files.
### ==============================================================================

In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import os
from pathlib import Path
from datetime import datetime

# --- CONFIGURATION ---
RAW_ROOT_DIR = Path(r'C:\Users\ASUS\Desktop\MVP\MVP_bioswot_m1')
OUT_DIR = Path(r'C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing\Data\raw')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. HEADER PARSING (METADATA EXTRACTION) ---

def parse_nmea_coord(nmea_str):
    """
    Converts NMEA string format (ddmm.mmmm) to Decimal Degrees.
    Returns np.nan if parsing fails.
    """ 
    try:
        parts = nmea_str.strip().split(',')
        val = float(parts[0])
        hemi = parts[1] if len(parts) > 1 else ''
        
        deg = int(val / 100)
        mins = val % 100
        decimal = deg + (mins / 60.0)
        
        if hemi in ['S', 'W']: decimal = -decimal
        return decimal
    except: return np.nan

def parse_m1_header(filepath):
    """
    Reads and extracts actual metadata from the .m1 file header.
    Looks for Latitude, Longitude, Date, and Time.
    """
    meta = {'lat': np.nan, 'lon': np.nan, 'datetime': None}
    try:
        with open(filepath, 'r', encoding='latin-1', errors='ignore') as f:
        # Read the first 60 lines where the header is located
            header_lines = [f.readline() for _ in range(60)]
        
        date_str, time_str = None, None
        
        for line in header_lines:
            line = line.strip()

            if "LAT" in line and ":" in line:
                meta['lat'] = parse_nmea_coord(line.split(':')[1])
            if "LON" in line and ":" in line:
                meta['lon'] = parse_nmea_coord(line.split(':')[1])

            if "Time (" in line and ":" in line:
                time_str = line.split(':', 1)[1].strip()
            if "Date (" in line and ":" in line:
                date_str = line.split(':')[1].strip()
                
        if date_str and time_str:
            try:
                # BioSWOT format: 22/04/2023 16:47:56.6
                dt_str = f"{date_str} {time_str}"
                meta['datetime'] = pd.to_datetime(dt_str, dayfirst=True)
            except: pass
            
    except Exception as e:
        pass 
        
    return meta

# --- 2. GENERIC NETCDF GENERATOR ---

def create_mvp_dataset(df, meta, filename, cast_type, station_name):
    if df.empty: return None

    # m1 has: Press, Depth, SV, Temp, Cond, Sal
    data_vars = {}
    
    if 'Press' in df: data_vars['pressure'] = (('scan',), df['Press'].values)
    if 'Temp' in df:  data_vars['t1'] = (('scan',), df['Temp'].values)
    if 'Cond' in df:  data_vars['c1'] = (('scan',), df['Cond'].values)
    if 'Sal' in df:   data_vars['s_raw'] = (('scan',), df['Sal'].values)
    
    if not data_vars: return None 

    ds = xr.Dataset(data_vars, coords={'scan': np.arange(len(df))})
    
    # Metadata
    if not np.isnan(meta['lat']):
        ds = ds.assign_coords(latitude=meta['lat'])
        ds['latitude'].attrs = {
            'units': 'degrees_north',
            'standard_name': 'latitude',
            'long_name': 'Latitude'
        }
    if not np.isnan(meta['lon']):
        ds = ds.assign_coords(longitude=meta['lon'])
        ds['longitude'].attrs = {
            'units': 'degrees_east',
            'standard_name': 'longitude',
            'long_name': 'Longitude'
        }
    if meta['datetime'] is not None:
        ds = ds.assign_coords(time=meta['datetime'])
        ds['time'].attrs = {
            'standard_name': 'time',
            'long_name': 'Time',
            'axis': 'T'
        }

    ds.attrs['title'] = f"MVP Profile {filename}"
    ds.attrs['source_file'] = filename
    ds.attrs['original_station'] = station_name
    ds.attrs['cast_direction'] = cast_type
    ds.attrs['instrument'] = 'Moving Vessel Profiler'
    ds.attrs['processing_level'] = 'Raw data converted from .m1'
    ds.attrs['history'] = f'Created {datetime.now().strftime("%Y-%m-%d")}; Split {cast_type}'
    ds.attrs['Conventions'] = 'CF-1.6'
    ds.attrs['processed_by'] = 'Elisabet Verger-Miralles'
    ds.attrs['ship'] = 'R/V Atlante'
    ds.attrs['cruise'] = 'BioSWOT-Med'

    if meta['datetime'] is not None:
        dt_utc = pd.to_datetime(meta['datetime'], utc=True)
        ds.attrs['date_utc'] = dt_utc.strftime('%Y-%m-%d')
        ds.attrs['time_utc'] = dt_utc.strftime('%H:%M:%S')
    else:
        ds.attrs['date_utc'] = 'unknown'
        ds.attrs['time_utc'] = 'unknown'
    
    # Variable metadata
    if 'pressure' in ds:
        ds['pressure'].attrs = {
            'units': 'dbar',
            'long_name': 'Pressure',
            'standard_name': 'sea_water_pressure',
            'axis': 'Z',
            'positive': 'down'
        }
    if 't1' in ds:
        ds['t1'].attrs = {
            'units': 'degC',
            'long_name': 'Temperature',
            'standard_name': 'sea_water_temperature'
        }
    if 'c1' in ds:
        ds['c1'].attrs = {
            'units': 'mS/cm',
            'long_name': 'Conductivity',
            'standard_name': 'sea_water_electrical_conductivity'
        }
    if 's_raw' in ds:
        ds['s_raw'].attrs = {
            'units': 'PSU',
            'long_name': 'Raw Salinity (Instrument)',
            'standard_name': 'sea_water_practical_salinity'
        }
    
    return ds

# --- 3. MAIN PROCESSING ROUTINE ---

def process_m1_file(filepath):
    """
    Processes a single .m1 file: parses metadata, reads data, splits into 
    downcast/upcast, and saves to NetCDF.
    """
    filename = filepath.name
    station_folder = filepath.parent.name
    file_stem = f"{station_folder}_{filepath.stem}" 
    
    # A) Read Metadata from Header
    meta = parse_m1_header(filepath)
    
    # B) Read Data Body
    try:
        header_row_idx = None
        with open(filepath, 'r', encoding='latin-1') as f:
            for i, line in enumerate(f):
                # Header starts as "Press" y tiene "Temp"
                if line.strip().startswith("Press") and "Temp" in line:
                    header_row_idx = i
                    break
        
        if header_row_idx is None:
            return

        # C) Read CSV
        #  Units are in next row of the heading

        df = pd.read_csv(filepath, skiprows=header_row_idx, 
                         encoding='latin-1', sep=r',\s*|\s+', engine='python')
        
    
        if isinstance(df.iloc[0,0], str) and 'dbar' in str(df.iloc[0,0]):
            df = df.iloc[1:].reset_index(drop=True)
            
        df = df.apply(pd.to_numeric, errors='coerce')
        df = df.dropna(subset=['Press']) # remove empty rows
        
    except Exception as e:
        print(f"❌ Error reading data in {filename}: {e}")
        return

    if df.empty: return

    # D) Split Down/Up
    try:
        idx_max = df['Press'].idxmax()
        df_down = df.iloc[0 : idx_max + 1]
        
        # Save only Downcast 
        if not df_down.empty:
            ds_down = create_mvp_dataset(df_down, meta, filename, 'downcast', station_folder)
            if ds_down:
                ds_down.to_netcdf(OUT_DIR / f"{file_stem}_downcast.nc")
                
    except Exception as e:
        print(f"Error procesando dataframe {filename}: {e}")

# --- 4. EXECUTION ---

print(f"--- Processsing BIOSWOT since {RAW_ROOT_DIR} ---")
files = sorted(list(RAW_ROOT_DIR.rglob("*.m1")) + list(RAW_ROOT_DIR.rglob("*.M1")))
print(f" {len(files)} files found.")

count = 0
for f in files:
    process_m1_file(f)
    count += 1
    if count % 100 == 0: print(f" ... {count}")

print(f"\n✅ END. Files saved in {OUT_DIR}")